# Coverage simplification and advanced routing lab

This lab connects two important ideas: preserving shared boundaries while simplifying polygon coverages, and accelerating shortest-path queries on directed graphs. The examples use small, reproducible data so you can inspect every assumption.

## 1. Simplify a polygon coverage

The two polygons share a deliberately wavy edge. Independent per-feature simplification can make adjacent copies diverge; coverage-aware simplification processes the shared boundary consistently.

In [ ]:
from pathlib import Path
import sys
sys.path.insert(0, str(Path.cwd() / 'implementations/python/coverage-simplification'))
sys.path.insert(0, str(Path.cwd() / 'implementations/python/precision-grid'))
from shapely.geometry import Polygon
from shapely import coverage_is_valid
from coverage_simplification import simplify_coverage

shared = [(5, 0), (6, 1), (4, 2), (6, 3), (4, 4), (6, 5), (4, 6), (6, 7), (5, 8)]
left = Polygon([(0, 0), *shared, (0, 8), (0, 0)])
right = Polygon([(5, 0), (10, 0), (10, 8), (5, 8), *reversed(shared[:-1])])
coverage = [left, right]
assert coverage_is_valid(coverage)
simplified = simplify_coverage(coverage, tolerance=1.5)
assert coverage_is_valid(simplified)
print('Vertices before:', [len(g.exterior.coords) for g in coverage])
print('Vertices after: ', [len(g.exterior.coords) for g in simplified])
print('Area before/after:', sum(g.area for g in coverage), sum(g.area for g in simplified))

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, geometries, title in zip(axes, [coverage, simplified], ['Original coverage', 'Coverage simplified']):
    for geom in geometries:
        x, y = geom.exterior.xy
        ax.plot(x, y, marker='o')
        ax.fill(x, y, alpha=0.15)
    ax.set_aspect('equal')
    ax.set_title(title)
plt.tight_layout()
plt.show()

## 2. Compare fixed-grid precision operations

Coordinate precision reduction and fixed-grid overlay are useful tools, but neither should be casually renamed as a complete snap-rounding implementation. Watch how a narrow polygon can collapse when the grid is too coarse.

In [ ]:
from shapely.geometry import LineString, Polygon
from precision_grid import reduce_precision, fixed_precision_intersection

first = LineString([(0.04, 0.04), (1.04, 1.04)])
second = LineString([(0.04, 1.04), (1.04, 0.04)])
print('Reduced coordinates:', list(reduce_precision(first, 0.1).coords))
print('Fixed-grid crossing:', fixed_precision_intersection(first, second, 0.1))
narrow = Polygon([(0, 0), (0.04, 0), (0.04, 1), (0, 1)])
print('Narrow polygon after 0.1 grid:', reduce_precision(narrow, 0.1).wkt)

## 3. Compare Dijkstra, bidirectional Dijkstra and ALT

The graph is directed. The reverse search in bidirectional Dijkstra must traverse reversed arcs; the ALT heuristic uses distances from and to landmarks. Every result should agree with Dijkstra's optimal cost.

In [ ]:
from pathlib import Path
import sys
sys.path.insert(0, str(Path.cwd() / 'implementations/python/shortest-paths'))
from shortest_paths import dijkstra, bidirectional_dijkstra, astar, ALTHeuristic

graph = {
    'A': [('B', 2), ('C', 1)],
    'B': [('D', 2), ('E', 2)],
    'C': [('B', 0.5), ('D', 8)],
    'D': [('F', 1)],
    'E': [('F', 1)],
    'F': []
}
reference = dijkstra(graph, 'A', 'F')
bidirectional = bidirectional_dijkstra(graph, 'A', 'F')
alt = ALTHeuristic(graph, landmarks=['A', 'F'])
landmark_path = astar(graph, 'A', 'F', alt)
print('Dijkstra:              ', reference)
print('Bidirectional Dijkstra:', bidirectional)
print('ALT A*:                ', landmark_path)
assert reference[1] == bidirectional[1] == landmark_path[1]

## 4. Investigate the trade-offs

1. Increase the number of vertices and edges. How does the bidirectional search frontier change?
2. Try different landmark sets. Does the ALT lower bound become tighter?
3. Add a one-way restriction by removing a reverse arc. Verify that the backward search still returns the correct directed path.
4. Introduce a turn restriction. Explain why the state may need to include the incoming edge rather than only the current node.
5. Change travel costs while keeping the graph topology fixed. Explain why ALT landmark distances may need to be recomputed.

These are teaching baselines, not a production road router. They do not model turn-expanded states, time-dependent costs, transit services or vehicle restrictions.